In [1]:
# DO NOT CONTAINERISE
# =====

# Dependency
# -----
# ! pip install -r requirements.txt
# ! pip list
# ! conda list

# !conda install -y requests
# !conda install -y nest_asyncio

# !conda install -y geopandas=0.10.2
# !conda install -y rdflib=6.1.1

# !pip show requests
# !pip show nest_asyncio

# !pip show geopandas
# !pip show rdflib

import os
import sys
import glob
import shutil
from pathlib import Path
from datetime import datetime

from io import StringIO
import zipfile
import asyncio
import requests
from urllib import parse
import json.decoder

import csv
import pandas as pd

import nest_asyncio

# base settings
# -----
conf_vlab_name     = "DNA"
conf_workflow_name = "PEMA"

# conf_workflow_id   = f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
param_workflow_name = "test"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# conf_dir_code = os.path.join("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, "Git public", "library")
# if not os.path.exists(conf_dir_code):
#     os.makedirs(conf_dir_code)

# conf_dir_data  = os.path.join("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, param_workflow_name)
# if not os.path.exists(conf_dir_data):
#     os.makedirs(conf_dir_data)

# local
# -----
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp = os.path.join("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      = "naa-vre-public"
conf_minio_public_bucket_root = f"vl-{conf_vlab_name.lower()}"
conf_minio_public_local_root  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data", conf_workflow_name)

conf_minio_user_bucket        = "naa-vre-user-data"
# conf_minio_user_bucket_root   = param_user_email
conf_minio_user_bucket_root   = conf_vlab_name
conf_minio_user_local_root    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   f"{conf_workflow_name}-{param_workflow_name}")
conf_minio_user_local_flog    = os.path.join(conf_minio_user_local_data, "log.md")

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# Input param
# -----
# workflow: 01, 02
# .....

# PEMA-SequenceRetriever
# .....
conf_fname_seq_zip = "mydata.zip"
conf_fpath_seq_zip = "mydata"                                                  # path in zip file

param_gene_sequences = "SRR3231901"
# param_gene_sequences = "ERR3460470,ERR4018451,ERR4018452"                    # user input, sep=","

# PEMA-Runner
# .....
# return: case_id
conf_fname_par_tsv  = "parameters.tsv"

param_fname_par_tsv = "Template-parameters.tsv"                                # upload file to conf_minio_user_local_root
# add param_ for allowed settings in pema parameters.tsv

# OTU
# .....
# pema_otu_delimiter = "\t"
# bold_otu_delimiter = ","
conf_delimiter_tsv = "\t"
conf_delimiter_csv = ","

print("Finish: NaaVRE parameters")
print(f"Workspace public:")
print(f"  Root: {conf_minio_public_local_root}")
print(f"  Code: {conf_minio_public_local_code}")
print(f"  Data: {conf_minio_public_local_data}")

print(f"Workspace user:")
print(f"  Root: {conf_minio_user_local_root}")
print(f"  Code: {conf_minio_user_local_code}")
print(f"  Data: {conf_minio_user_local_data}")
print(f"  Log:  {conf_minio_user_local_flog}")


Finish: NaaVRE parameters
Workspace public:
  Root: /home/jovyan/Cloud Storage/naa-vre-public/vl-dna
  Code: /home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code
  Data: /home/jovyan/Cloud Storage/naa-vre-public/vl-dna/data/PEMA
Workspace user:
  Root: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA
  Code: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/library
  Data: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test
  Log:  /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/log.md


## note

| Domain | Phylum | Class | Order | Family | Genus | Species |
| ------ | ------ | ----- | ----- | ------ | ----- | ------- |
| 域     | 门     | 纲     | 目    | 科     | 属    | 种      |

## test dataset

| id | dataset                                        | usage                                 | env | Local | NaaVRE | MyLifewatch      |
| -- | ---------------------------------------------- | ------------------------------------- | --- | ----- | ------ | ---------------- |
| 99 | SRR3231900,SRR3231901                          | PEMA,  Source code, v.2.1.4           | py  |       |        | fail: metamds    |
| 00 | ERR7125480,ERR7125483,ERR7125486,ERR7125489    | WRiMS, Invasive_Checker               | py  |       |        | fail: PEMARunner |
| 01 | ERR12541385,ERR12541386,ERR12541387,ERR7125481 | PEMA,  SequenceRetriever, mylifewatch | py  |       |        | fail: PEMARunner |
| 02 | ERR2181465,ERR2181466,ERR2181467               | PEMA,  Runner, OTU                    | py  |       |        | fail: PEMARunner |
| 03 | ERR3460470,ERR4018451,ERR4018452               | PEMA,  Converter                      | r   |       |        | fail: PEMARunner |

### 99, test
```python
param_gene_sequences = "SRR3231900,SRR3231901"
param_workflow_name  = "wid-test_dataset_99-PEMA_Source_code_v.2.1.4"
case_id              = "cid-test_dataset_99-PEMA_Source_code_v.2.1.4"
```

### 00
```python
param_gene_sequences = "ERR7125480,ERR7125483,ERR7125486,ERR7125489"
param_workflow_name  = "wid-test_dataset_00-WRiMS_Invasive_Checker"
case_id              = "cid-test_dataset_00-WRiMS_Invasive_Checker"
```

### 01
```python
param_gene_sequences = "ERR12541385,ERR12541386,ERR12541387,ERR7125481"
param_workflow_name  = "wid-test_dataset_01-PEMA_SequenceRetriever"
case_id              = "cid-test_dataset_01-PEMA_SequenceRetriever"
```

### 02
```python
param_gene_sequences = "ERR2181465,ERR2181466,ERR2181467"
param_workflow_name  = "wid-test_dataset_02-PEMA_Runner-OTU"
case_id              = "cid-test_dataset_02-PEMA_Runner-OTU"
```

### 03
```python
param_gene_sequences = "ERR3460470,ERR4018451,ERR4018452"
param_workflow_name  = "wid-test_dataset_03-PEMA_Converter"
case_id              = "cid-test_dataset_03-PEMA_Converter"
```


In [2]:
# DNA, workflow start
# ---
# NaaVRE:
#  cell:
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

# prepare folders
# .....
if not os.path.exists(conf_dir_data_local_tmp):
    os.makedirs(conf_dir_data_local_tmp)

# if not os.path.exists(conf_minio_public_local_root):
#     os.makedirs(conf_minio_public_local_root)

if not os.path.exists(conf_minio_user_local_root):
    os.makedirs(conf_minio_user_local_root)

if not os.path.exists(conf_minio_user_local_data):
    os.makedirs(conf_minio_user_local_data)
    
with open(conf_minio_user_local_flog, "w+") as fp_log:
    fp_log.write(f"# {param_workflow_name}\n")

# create log
# .....
print(param_workflow_name)
print(param_gene_sequences)
print(param_fname_par_tsv)
workflow_step = f"{conf_vlab_name}-Start"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n")
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n")

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")


test
SRR3231901
Template-parameters.tsv
Finish: DNA-Start


In [3]:
# DNA, PEMA-SequenceRetriever
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# workflow: 01, 02, 03

# test in terminal
# -----
# python /home/jovyan/library/enaBrowserTools/python3_1.6/enaDataGet.py -f fastq -d /tmp/data/ena ERR12541385

import os
import sys
import glob

import zipfile

print(param_workflow_name)
print(param_gene_sequences)
workflow_step = f"{conf_vlab_name}-PEMA_SequenceRetriever"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)
dir_lib_ena = os.path.join(conf_minio_public_local_code,  "enaBrowserTools", "python3_1.6")
fname_lib_ena = "enaDataGet.py"
file_lib_ena = os.path.join(dir_lib_ena,  fname_lib_ena)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

dir_pema = os.path.join(conf_minio_user_local_data, workflow_step)

dir_pema_ena = os.path.join(dir_pema)
if not os.path.exists(dir_pema_ena):
    os.makedirs(dir_pema_ena)

dir_pema_seq = os.path.join(conf_minio_user_local_data)
if not os.path.exists(dir_pema_seq):
    os.makedirs(dir_pema_seq)

fname_mydata_zip = f"{conf_fname_seq_zip}"
file_seq_zip     = os.path.join(dir_pema_seq, fname_mydata_zip)

# func
# -----

# start
# -----
genes_list = param_gene_sequences.split(',') if len(param_gene_sequences) else []

if (len(genes_list) == 0):
    raise RuntimeError("At least 1 gene sequence must be provided")

for gene in genes_list:
    # Run ENA browser tools
    # os.system(f"python {file_lib_ena} -f fastq -d {dir_pema_ena} {gene}")
    os.system(f'python "{file_lib_ena}" -f fastq -d "{dir_pema_ena}" -c {gene}')
    
    if len(os.listdir(os.path.join(dir_pema_ena, gene))) != 2:
        raise RuntimeError(f"Expected 2 files for gene `{gene}`. Found {len(os.listdir(dir_pema_ena))} @ {dir_pema_ena}")

file_fastq_list = glob.glob(os.path.join(dir_pema_ena, "*", "*.fastq.gz"))

with zipfile.ZipFile(file_seq_zip, "w") as zipObj:
    for file_fastq in file_fastq_list:
        zipObj.write(file_fastq, os.path.join(conf_fpath_seq_zip, os.path.basename(file_fastq)))
        print(f"Info: Compress gene file {file_fastq}")

print(f"Info: Generated compressed sequencing files {file_fastq}")

if not os.path.isfile(file_seq_zip):
    raise FileNotFoundError(str(file_seq_zip) + " is missing")

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {file_seq_zip}\n")

print(f"Finish: {workflow_step}")


test
SRR3231901


/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTools/python3_1.6/utils.py:103: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  sequence_pattern_1 = re.compile('^[A-Z]{1}[0-9]{5}(\.[0-9]+)?$')
/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTools/python3_1.6/utils.py:104: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  sequence_pattern_2 = re.compile('^[A-Z]{2}[0-9]{6}(\.[0-9]+)?$')
/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTools/python3_1.6/utils.py:105: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  wgs_sequence_pattern = re.compile('^[A-Z]{4}[0-9]{8,9}(\.[0-9]+)?$')
/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTo

Checking availability of https://www.ebi.ac.uk/ena/browser/api/xml/SRR3231901
SRR3231901_1.fastq.gz already exists in local directory, skipping
SRR3231901_2.fastq.gz already exists in local directory, skipping
Completed
Info: Compress gene file /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/PEMA-SequenceRetriever/SRR3231901/SRR3231901_1.fastq.gz
Info: Compress gene file /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/PEMA-SequenceRetriever/SRR3231901/SRR3231901_2.fastq.gz
Info: Generated compressed sequencing files /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/PEMA-SequenceRetriever/SRR3231901/SRR3231901_2.fastq.gz
Finish: PEMA-SequenceRetriever


In [4]:
# DNA, PEMA-parameters
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile

print(param_workflow_name)
print(param_fname_par_tsv)
workflow_step_i = f"{conf_vlab_name}-PEMA_SequenceRetriever"
workflow_step   = f"{conf_vlab_name}-PEMA_Parameters"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

file_param_tsv_i = os.path.join(conf_minio_user_local_root, param_fname_par_tsv)

# output
# -----
dummy_cell_arg_o = "dummy output"

file_param_tsv_o = os.path.join(conf_minio_user_local_data, conf_fname_par_tsv)

# func
# -----

# start
# -----
# set parameters
# .....
with open(file_param_tsv_i, 'r') as fp_i, open(file_param_tsv_o, 'w') as fp_o:
    for line_i in fp_i:
        line_o = line_i.strip()
        fp_o.write(f"{line_o}\n")

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {file_param_tsv_o}\n")

print(f"Finish: {workflow_step}")


test
Template-parameters.tsv
Finish: PEMA-parameters


In [17]:
# DO NOT CONTAINERISE
# =====

# test api
# .....
url = "https://pema-dev.naavre.net/pema/case/"
obj_response = requests.get(url)
dict_response = obj_response.json()
print(obj_response.status_code)
for key, val in dict_response.items():
    print("\n", key, "\n")
    for val_sub in val:
        print(val_sub, "\n")


200

 20260407_123714994975 

{'filename': 'Result', 'last_modified': '2026-04-07 12:45:46', 'extension': 'unknown'} 

{'filename': 'mydata', 'last_modified': '2026-04-07 12:37:23', 'extension': 'unknown'} 

{'filename': 'parameters.tsv', 'last_modified': '2026-04-07 12:37:15', 'extension': '.tsv'} 

{'filename': 'PEMA-20260407_123714994975.log', 'last_modified': '2026-04-07 12:45:46', 'extension': '.log'} 

{'filename': 'PEMA-20260407_123714994975.zip', 'last_modified': '2026-04-07 12:45:47', 'extension': '.zip'} 


 20260407_122733918750 

{'filename': 'Result', 'last_modified': '2026-04-07 12:36:05', 'extension': 'unknown'} 

{'filename': 'mydata', 'last_modified': '2026-04-07 12:27:42', 'extension': 'unknown'} 

{'filename': 'PEMA-20260407_122733918750.zip', 'last_modified': '2026-04-07 12:36:05', 'extension': '.zip'} 

{'filename': 'parameters.tsv', 'last_modified': '2026-04-07 12:27:34', 'extension': '.tsv'} 

{'filename': 'PEMA-20260407_122733918750.log', 'last_modified': '2026-

In [5]:
# DNA, PEMA-Runner
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# On UvA VM, https://pema-dev.naavre.net
# Codes are in README.md, tested on NaaVRE openLab

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile
import requests

print(param_workflow_name)
workflow_step_i = f"{conf_vlab_name}-PEMA_Parameters"
workflow_step   = f"{conf_vlab_name}-PEMA_Runner"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

file_param_tsv_o = os.path.join(conf_minio_user_local_data, conf_fname_par_tsv)

dir_upload_zip   = os.path.join(conf_minio_user_local_data)
fname_mydata_zip = f"{conf_fname_seq_zip}"
file_upload_zip  = os.path.join(dir_upload_zip, fname_mydata_zip)

# output
# -----
dummy_cell_arg_o = "dummy output"

# case_id   = "cid-test_dataset_99-PEMA_Source_code_v.2.1.4"
case_id   = ""  # request from API
case_name = f"PEMA-{case_id}"

dir_pema = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_pema):
    os.makedirs(dir_pema)

fname_result_zip  = f"{case_name}.zip"
file_download_zip = os.path.join(dir_pema, fname_result_zip)

dir_unzip = os.path.join(conf_minio_user_local_data, workflow_step, "result")
if not os.path.exists(dir_unzip):
    os.makedirs(dir_unzip)

# func
# -----

# start
# -----
# zip "parameters.tsv" and "mydata" into "mydata.zip"
# .....
# append "parameters.tsv" into "mydata.zip"
with zipfile.ZipFile(file_upload_zip, "a") as zipObj:
    zipObj.write(file_param_tsv_o, os.path.basename(file_param_tsv_o))

# api
# .....
with open(file_upload_zip, 'rb') as fp_upload_zip, open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = f"Start: PEMA API, at {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
    print(str_print)
    fp_log.write(f"{str_print}\n")

    # Upload mydata and param
    # .....
    # output: case_id = f"cid-{}"
    url = "https://pema-dev.naavre.net/pema/case/"

    files = {'case_zip_file': (fname_mydata_zip, fp_upload_zip)}
    obj_response = requests.post(url, files=files)
    if obj_response.status_code == 200:
        dict_response = obj_response.json()
        case_id   = dict_response["case_id"]
        case_name = f"PEMA-{case_id}"
    
        str_print = f"Finish: Upload mydata and param, {case_name}, at {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
        print(str_print)
        fp_log.write(f"{str_print}\n")
    else:
        print(obj_response.status_code)
        print(obj_response.text)

    # Run pema
    # .....
    # input: case_id
    url = f"https://pema-dev.naavre.net/pema/run/?case_id={case_id}"
    is_pema_run = False
    
    obj_response = requests.post(url, timeout=None)
    if obj_response.status_code == 200:
        # dict_response = obj_response.json()
        is_pema_run = True
        
        str_print = f"Finish: Run pema, {case_name}, at {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
        print(str_print)
        fp_log.write(f"{str_print}\n")
    else:
        print(obj_response.status_code)
        print(obj_response.text)

    # Get pema result
    # .....
    # input: case_id
    url = f"https://pema-dev.naavre.net/pema/result/?case_id={case_id}"

    if is_pema_run:
        fname_result_zip  = f"{case_name}.zip"
        file_download_zip = os.path.join(dir_pema, fname_result_zip)
        
        # with zipfile.ZipFile(file_download_zip, 'r') as zipObj:
        #     zipObj.extractall(dir_unzip)
        
        obj_response = requests.get(url, stream=True)
        if obj_response.status_code == 200:
            # dict_response = obj_response.json()
            
            chunk_size = 128
            with open(file_download_zip, 'wb') as fd:
                for chunk in obj_response.iter_content(chunk_size=chunk_size):
                    fd.write(chunk)
        
            if os.path.isfile(file_download_zip):
                with zipfile.ZipFile(file_download_zip, 'r') as zip_ref:
                    zip_ref.extractall(dir_unzip)
                
                str_print = f"Finish: Get pema result, {case_name}, at {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
                print(str_print)
                fp_log.write(f"{str_print}\n")
                
                str_print = f"Output: {dir_unzip}"
                print(str_print)
                fp_log.write(f"{str_print}\n")
            else:
                raise FileNotFoundError(f"{file_download_zip} is missing")
        else:
            print(obj_response.status_code)
            print(obj_response.text)

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")

print(f"Finish: {workflow_step}")


test
Start: PEMA API, PEMA-, 20260409_083813507858
Finish: Upload mydata and param, PEMA-20260409_083813978000, 20260409_083814093610
Finish: Run pema, PEMA-20260409_083813978000, 20260409_084701847532
Finish: Get pema result, PEMA-20260409_083813978000, 20260409_084702559920
Output: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/PEMA-Runner/result
Finish: PEMA-Runner


In [6]:
# DNA, OTU-Unify
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile
import requests
from pathlib import Path
import pandas as pd

print(param_workflow_name)
workflow_step_i = f"{conf_vlab_name}-PEMA_Runner"
workflow_step   = f"{conf_vlab_name}-OTU_Unify"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

dir_pema = os.path.join(conf_minio_user_local_data, workflow_step_i, "result")
print(dir_pema)

# file_i_pema_otu = os.path.join(dir_pema, 'final_table.tsv')
file_i_pema_otu = os.path.join(dir_pema, 'finalTable.tsv')
file_i_bold_otu = os.path.join(dir_pema, 'all_sequences_grouped.fa_output.csv')

# output
# -----
dummy_cell_arg_o = "dummy output"

dir_otu = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_otu):
    os.makedirs(dir_otu)

# file_o_otu_csv  = Path(os.path.join(dir_otu,  'otu.tsv'))
# file_o_otu_csv.touch()
file_o_otu_csv  = os.path.join(dir_otu,  'otu.tsv')

# func
# -----

# start
# -----
is_pema_exists = Path(file_i_pema_otu).is_file()
is_bold_exists = Path(file_i_bold_otu).is_file()

if not is_pema_exists:
    raise RuntimeError('It is mandatory to provide the PEMA TSV file.')

# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    df_pema = pd.read_csv(file_i_pema_otu, sep=conf_delimiter_tsv)

    # print("DevOps: df_pema-00")
    # str_print = f"{df_pema}"
    # print(str_print)
    # fp_log.write(f"\n```\n{str_print}\n```\n\n")
    
    if "Classification" in df_pema.columns:
        df_pema = df_pema.rename(columns={"Classification": "classification"})
    
    # rename first column to `OTU`. In the case of PEMA its called `Otuamplicon`
    df_pema = df_pema.rename(columns={df_pema.columns[0]: "OTU"})
    
    # otu_df["classification"] = otu_df["classification"].apply(lambda x: str(x).split(";")[-1])
    
    missing_row_indexes = df_pema.loc[
        (df_pema['classification'].str.contains("No hits"))
        | (df_pema['classification'].str.contains("Unknown"))
        | (df_pema['classification'].str.contains("root"))
    ].index
    # print("DevOps: missing_row_indexes")
    # print(missing_row_indexes)
    
    if df_pema["OTU"].str.startswith("ASV").all():
        df_pema["OTU"] = df_pema["OTU"].str.split(":", expand=True)[1]
    
    df_pema.drop(missing_row_indexes, inplace=True)

    # print("DevOps: df_pema-01")
    # str_print = f"{df_pema.describe()}"

    str_print = f"{df_pema}"
    print(str_print)
    fp_log.write(f"\n```\n{str_print}\n```\n\n")

# .....
if not is_bold_exists:
    df_pema.to_csv(file_o_otu_csv, sep=conf_delimiter_tsv, index=False, mode="w")

else:
    df_bold = pd.read_csv(file_i_bold_otu, sep=conf_delimiter_csv)
    df_bold = df_bold.rename(columns={"OtuID": "OTU"})

    df_bold["classification"] = (
        df_bold["phylum"].astype(str)
        + ";"
        + df_bold["class"].astype(str)
        + ";"
        + df_bold["order"].astype(str)
        + ";"
        + df_bold["family"].astype(str)
        + ";"
        + df_bold["genus"].astype(str)
        + ";"
        + df_bold["species"].astype(str)
    )

    missing_row_indexes = df_bold.loc[
        (df_bold["classification"].str.contains("nohit"))
    ].index

    df_bold.drop(missing_row_indexes, inplace=True)

    df_pema.drop("classification", inplace=True, axis=1)
    df_pema["OTU"] = df_pema["OTU"].str.replace("Otu", "")

    df_bold = df_bold[["ID", "OTU", "classification"]]
    df_bold["OTU"] = df_bold["OTU"].str.split("_", expand=True)[0]

    df_bold = pd.merge(df_pema, df_bold, on="OTU")
    df_bold.drop("OTU", inplace=True, axis=1)
    a = df_bold["ID"]
    df_bold.drop(labels=["ID"], axis=1, inplace=True)
    df_bold.insert(0, "ID", a)

    df_bold = df_bold.rename(columns={"ID": "OTU"})
    df_bold.to_csv(file_o_otu_csv, sep=conf_delimiter_tsv, index=False, mode="w")

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {dir_otu}\n")

print(f"Finish: {workflow_step}")


test
/home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/PEMA-Runner/result
    OTU  SRR3231901                                     classification
0  Otu8           2  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...
1  Otu6          13  Main genome;Eukaryota;Cryptophyceae;Cryptomona...
2  Otu7           3  Chloroplast;Eukaryota (Chloroplast);Archaeplas...
3  Otu4          28  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...
4  Otu5          18  Chloroplast;Eukaryota (Chloroplast);Archaeplas...
5  Otu2          46  Main genome;Eukaryota;SAR;Alveolata;Dinoflagel...
6  Otu3          27  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...
7  Otu1          88  Main genome;Eukaryota;SAR;Alveolata;Dinoflagel...
Finish: OTU-Unify


In [16]:
# DNA, WoRMS-Taxonomic Checker wrapper 
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile
import asyncio
import asyncio.events as events
import requests
from urllib import parse
import json.decoder

from pathlib import Path
import pandas as pd

# try:
#     import nest_asyncio
# else:
#     import nest-asyncio as nest_asyncio
# nest_asyncio.apply()
import threading
from contextlib import contextmanager, suppress
from heapq import heappop

print(param_workflow_name)
workflow_step_i = f"{conf_vlab_name}-OTU_Unify"
workflow_step   = f"{conf_vlab_name}-WoRMS"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

dir_otu = os.path.join(conf_minio_user_local_data, workflow_step_i)
fname_worms_i = 'otu.tsv'
file_worms_i  = os.path.join(dir_otu, fname_worms_i)

# output
# -----
dummy_cell_arg_o = "dummy output"

dir_worms = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_worms):
    os.makedirs(dir_worms)

fname_worms_o = 'worms.tsv'
file_worms_o  = os.path.join(dir_worms, fname_worms_o)

# func
# -----
# nest_asyncio.apply()
# .....
def nest_asyncio_apply(loop=None):
    """Patch asyncio to make its event loop reentrant."""
    _patch_asyncio()
    _patch_policy()
    _patch_tornado()

    loop = loop or asyncio.get_event_loop()
    _patch_loop(loop)


def _patch_asyncio():
    """Patch asyncio module to use pure Python tasks and futures."""

    def run(main, *, debug=False):
        loop = asyncio.get_event_loop()
        loop.set_debug(debug)
        task = asyncio.ensure_future(main)
        try:
            return loop.run_until_complete(task)
        finally:
            if not task.done():
                task.cancel()
                with suppress(asyncio.CancelledError):
                    loop.run_until_complete(task)

    def _get_event_loop(stacklevel=3):
        loop = events._get_running_loop()
        if loop is None:
            loop = events.get_event_loop_policy().get_event_loop()
        return loop

    # Use module level _current_tasks, all_tasks and patch run method.
    if hasattr(asyncio, '_nest_patched'):
        return
    if sys.version_info >= (3, 6, 0):
        asyncio.Task = asyncio.tasks._CTask = asyncio.tasks.Task = \
            asyncio.tasks._PyTask
        asyncio.Future = asyncio.futures._CFuture = asyncio.futures.Future = \
            asyncio.futures._PyFuture
    if sys.version_info < (3, 7, 0):
        asyncio.tasks._current_tasks = asyncio.tasks.Task._current_tasks
        asyncio.all_tasks = asyncio.tasks.Task.all_tasks
    if sys.version_info >= (3, 9, 0):
        events._get_event_loop = events.get_event_loop = \
            asyncio.get_event_loop = _get_event_loop
    asyncio.run = run
    asyncio._nest_patched = True


def _patch_policy():
    """Patch the policy to always return a patched loop."""

    def get_event_loop(self):
        if self._local._loop is None:
            loop = self.new_event_loop()
            _patch_loop(loop)
            self.set_event_loop(loop)
        return self._local._loop

    policy = events.get_event_loop_policy()
    policy.__class__.get_event_loop = get_event_loop


def _patch_loop(loop):
    """Patch loop to make it reentrant."""

    def run_forever(self):
        with manage_run(self), manage_asyncgens(self):
            while True:
                self._run_once()
                if self._stopping:
                    break
        self._stopping = False

    def run_until_complete(self, future):
        with manage_run(self):
            f = asyncio.ensure_future(future, loop=self)
            if f is not future:
                f._log_destroy_pending = False
            while not f.done():
                self._run_once()
                if self._stopping:
                    break
            if not f.done():
                raise RuntimeError(
                    'Event loop stopped before Future completed.')
            return f.result()

    def _run_once(self):
        """
        Simplified re-implementation of asyncio's _run_once that
        runs handles as they become ready.
        """
        ready = self._ready
        scheduled = self._scheduled
        while scheduled and scheduled[0]._cancelled:
            heappop(scheduled)

        timeout = (
            0 if ready or self._stopping
            else min(max(
                scheduled[0]._when - self.time(), 0), 86400) if scheduled
            else None)
        event_list = self._selector.select(timeout)
        self._process_events(event_list)

        end_time = self.time() + self._clock_resolution
        while scheduled and scheduled[0]._when < end_time:
            handle = heappop(scheduled)
            ready.append(handle)

        for _ in range(len(ready)):
            if not ready:
                break
            handle = ready.popleft()
            if not handle._cancelled:
                # preempt the current task so that that checks in
                # Task.__step do not raise
                curr_task = curr_tasks.pop(self, None)

                try:
                    handle._run()
                finally:
                    # restore the current task
                    if curr_task is not None:
                        curr_tasks[self] = curr_task

        handle = None

    @contextmanager
    def manage_run(self):
        """Set up the loop for running."""
        self._check_closed()
        old_thread_id = self._thread_id
        old_running_loop = events._get_running_loop()
        try:
            self._thread_id = threading.get_ident()
            events._set_running_loop(self)
            self._num_runs_pending += 1
            if self._is_proactorloop:
                if self._self_reading_future is None:
                    self.call_soon(self._loop_self_reading)
            yield
        finally:
            self._thread_id = old_thread_id
            events._set_running_loop(old_running_loop)
            self._num_runs_pending -= 1
            if self._is_proactorloop:
                if (self._num_runs_pending == 0
                        and self._self_reading_future is not None):
                    ov = self._self_reading_future._ov
                    self._self_reading_future.cancel()
                    if ov is not None:
                        self._proactor._unregister(ov)
                    self._self_reading_future = None

    @contextmanager
    def manage_asyncgens(self):
        if not hasattr(sys, 'get_asyncgen_hooks'):
            # Python version is too old.
            return
        old_agen_hooks = sys.get_asyncgen_hooks()
        try:
            self._set_coroutine_origin_tracking(self._debug)
            if self._asyncgens is not None:
                sys.set_asyncgen_hooks(
                    firstiter=self._asyncgen_firstiter_hook,
                    finalizer=self._asyncgen_finalizer_hook)
            yield
        finally:
            self._set_coroutine_origin_tracking(False)
            if self._asyncgens is not None:
                sys.set_asyncgen_hooks(*old_agen_hooks)

    def _check_running(self):
        """Do not throw exception if loop is already running."""
        pass

    if hasattr(loop, '_nest_patched'):
        return
    if not isinstance(loop, asyncio.BaseEventLoop):
        raise ValueError('Can\'t patch loop of type %s' % type(loop))
    cls = loop.__class__
    cls.run_forever = run_forever
    cls.run_until_complete = run_until_complete
    cls._run_once = _run_once
    cls._check_running = _check_running
    cls._check_runnung = _check_running  # typo in Python 3.7 source
    cls._num_runs_pending = 1 if loop.is_running() else 0
    cls._is_proactorloop = (
        os.name == 'nt' and issubclass(cls, asyncio.ProactorEventLoop))
    if sys.version_info < (3, 7, 0):
        cls._set_coroutine_origin_tracking = cls._set_coroutine_wrapper
    curr_tasks = asyncio.tasks._current_tasks \
        if sys.version_info >= (3, 7, 0) else asyncio.Task._current_tasks
    cls._nest_patched = True


def _patch_tornado():
    """
    If tornado is imported before nest_asyncio, make tornado aware of
    the pure-Python asyncio Future.
    """
    if 'tornado' in sys.modules:
        import tornado.concurrent as tc  # type: ignore
        tc.Future = asyncio.Future
        if asyncio.Future not in tc.FUTURES:
            tc.FUTURES += (asyncio.Future,)


# .....
async def aphiaID_by_name(classification, move_up_in_tree, url):
    name = classification.split(";")
    url_param = "marine_only=false"

    if move_up_in_tree:
        # reverse list
        name = name[::-1]
    else:
        # use only the last one in the list
        name = [name[-1]]

    # .....
    for n in name:
        url = f"{url}/{parse.quote(n)}?{url_param}"
        resp = requests.get(url)

        with open(conf_minio_user_local_flog, "a+") as fp_log:
            try:
                id = int(resp.text)
                str_print = f"aphia id `{id}` found for `{n}` .({url})"
                print(str_print)
                fp_log.write(f"* {str_print}\n")
                
                return {"classification": classification, "aphia_id": id}
            except ValueError:
                str_print = f"no aphia id found for `{n}`. ({url})"
                print(str_print)
                fp_log.write(f"* {str_print}\n")
                
                continue

    return {"classification": classification, "aphia_id": None}


async def aphia_distributions_by_aphia_id(aphia_id, url):
    url = f"{url}/{aphia_id}"
    resp = requests.get(url)

    try:
        r = resp.json()
        location_set = set()

        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"Distribution found for aphia id `{aphia_id}`. ({url})"
            print(str_print)
            fp_log.write(f"  * {str_print}\n")

        for item in r:
            if item["establishmentMeans"] == "Alien":
                location_id = item["locationID"]
                location_id = location_id.split("/")[-1]
                location_set.add(location_id)
        return {
            "aphia_id": [aphia_id] * len(location_set),
            "location_id": list(location_set),
        }

    except json.decoder.JSONDecodeError:
        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"No distribution found for aphia id `{aphia_id}`. ({url})"
            print(str_print)
            fp_log.write(f"  * {str_print}\n")

        return {"aphia_id": [aphia_id], "location_id": [None]}
    except Exception as err:
        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"Raised silent exception {err} while requesting {url}"
            print(str_print)
            fp_log.write(f"  * {str_print}\n")

        return {"aphia_id": [aphia_id], "location_id": [None]}


async def start():
    classification_move_up = True

    # file_worms_i = '/mnt/inputs/sample.csv'
    # file_worms_o = '/mnt/outputs/worms.csv'

    df_otu_table = pd.read_csv(file_worms_i, delimiter=conf_delimiter_tsv)

    rows, _ = df_otu_table.shape

    # .....
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        str_print = f"### Removing missing No hits/Uknown from:"
        print(str_print)
        fp_log.write(f"{str_print}\n")

        str_print = f"{df_otu_table}"
        print(str_print)
        fp_log.write(f"\n```\n{str_print}\n```\n\n")

        str_print = f"N-rows: {rows}"
        print(str_print)
        fp_log.write(f"{str_print}\n")

    tasks = []

    for classification in set(df_otu_table.classification.values):
        tasks.append(
            asyncio.ensure_future(
                aphiaID_by_name(classification, move_up_in_tree=classification_move_up, url=url_AphiaIDByName)
            )
        )

    results = await asyncio.gather(*tasks)

    try:
        df_otu_table = df_otu_table.merge(
            pd.DataFrame(results, columns=["classification", "aphia_id"]).set_index(
                "classification"
            ),
            on="classification",
        )
        rows, _ = df_otu_table.shape

        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"{df_otu_table}"
            print(str_print)
            fp_log.write(f"\n```\n{str_print}\n```\n\n")

            str_print = f"N-rows: {rows}"
            print(str_print)
            fp_log.write(f"{str_print}\n")

        # df_otu_table = df_otu_table.dropna()
        # rows, _ = df_otu_table.shape
        # print("Removing rows with missing aphia ids")
        # print(f"{df_otu_table}")
        # print(f"N-rows: {rows}")

    except ValueError:
        raise

    # .....
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        # df_otu_table["aphia_id"] = df_otu_table["aphia_id"].astype("int64")

        str_print = f"Running `AphiaDistributionsByAphiaID` for `{len(df_otu_table.aphia_id.values)}` aphia ids retrieved."
        print(str_print)
        fp_log.write(f"### {str_print}\n")

    tasks.clear()

    for aphia_id in set(df_otu_table.aphia_id.values):
        tasks.append(
            asyncio.ensure_future(aphia_distributions_by_aphia_id(aphia_id, url=url_AphiaDistributionsByAphiaID))
        )

    results = await asyncio.gather(*tasks)

    distribution_table = pd.DataFrame(columns=["aphia_id", "location_id"])

    # .....
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        str_print = f"Get aphia ids & rename"
        print(str_print)
        fp_log.write(f"### {str_print}\n")

    for res in results:
        part = pd.DataFrame(
            {"aphia_id": res["aphia_id"], "location_id": res["location_id"]}
        )
        distribution_table = pd.concat([distribution_table, part])

    try:
        # print("Removing rows with missing location ids")
        df_otu_table = df_otu_table.merge(distribution_table, on="aphia_id")

        # df_otu_table = df_otu_table.dropna()
        # print(f"{df_otu_table}")
        # print(f"N-rows: {rows}")
    except ValueError:
        raise

    if "location_id" not in df_otu_table.columns:
        df_otu_table["location_id"] = ""
    
    # it may be the case that some rows have the same ids,
    # in that case rename them in order for rv lab to work
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        dupls = df_otu_table.loc[df_otu_table["OTU"].duplicated(), :].index

        str_print = f"Renaming {len(dupls)} rows ids"
        print(str_print)
        fp_log.write(f"* {str_print}\n")

    i = 0
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        for dupl in dupls:
            n_name = df_otu_table.loc[dupl, "OTU"] + "_" + str(i)
            
            str_print = f"Renaming `{df_otu_table.loc[dupl, 'OTU']}` to `{n_name}`"
            print(str_print)
            fp_log.write(f"* {str_print}\n")

            df_otu_table.loc[dupl, "OTU"] = n_name
            i = i + 1

    # save to file
    # .....
    with open(file_worms_o, 'w') as file:
        df_otu_table.to_csv(file, index=False, sep=conf_delimiter_tsv)


# start
# -----
aphia_ids = []

url_AphiaIDByName               = "http://www.marinespecies.org/rest/AphiaIDByName"
url_AphiaDistributionsByAphiaID = "http://www.marinespecies.org/rest/AphiaDistributionsByAphiaID"

# Call main
# -----
# TODO, QPan, loop.close(): raise RuntimeError("Cannot close a running event loop")
# loop = asyncio.get_event_loop()
# try:
#     loop.run_until_complete(main())
# except Exception as err:
#     print("Error: ", err)
# finally:
#     loop.run_until_complete(loop.shutdown_asyncgens())
#     print(loop)
#     loop.close()

nest_asyncio_apply()

asyncio.run(start())

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {dir_worms}\n")

print(f"Finish: {workflow_step}")


test
### Removing missing No hits/Uknown from:
    OTU  SRR3231901                                     classification
0  Otu8           2  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...
1  Otu6          13  Main genome;Eukaryota;Cryptophyceae;Cryptomona...
2  Otu7           3  Chloroplast;Eukaryota (Chloroplast);Archaeplas...
3  Otu4          28  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...
4  Otu5          18  Chloroplast;Eukaryota (Chloroplast);Archaeplas...
5  Otu2          46  Main genome;Eukaryota;SAR;Alveolata;Dinoflagel...
6  Otu3          27  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...
7  Otu1          88  Main genome;Eukaryota;SAR;Alveolata;Dinoflagel...
N-rows: 8
aphia id `106289` found for `Rhodomonas` .(http://www.marinespecies.org/rest/AphiaIDByName/Rhodomonas?marine_only=false)
aphia id `148934` found for `Thalassiosira pseudonana` .(http://www.marinespecies.org/rest/AphiaIDByName/Thalassiosira%20pseudonana?marine_only=false)
no aphia id found for `Trebouxio

In [8]:
# DNA, WRiMS-Invasive Checker
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
import shutil
from pathlib import Path
from datetime import datetime

from io import StringIO
import zipfile
import asyncio
import requests
from urllib import parse
import json.decoder

import csv
import pandas as pd

print(param_workflow_name)
workflow_step_i = f"{conf_vlab_name}-WoRMS"
workflow_step   = f"{conf_vlab_name}-WRiMS"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)
dir_lib_InvasiveChecker   = os.path.join(conf_minio_public_local_code,  "invasive_checker")
fname_lib_InvasiveChecker = "main.py"
file_lib_InvasiveChecker  = os.path.join(dir_lib_InvasiveChecker,  fname_lib_InvasiveChecker)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

dir_worms = os.path.join(conf_minio_user_local_data, workflow_step_i)
if not os.path.exists(dir_worms):
    os.makedirs(dir_worms)

fname_i_worms = 'worms.tsv'
file_i_worms  = os.path.join(dir_worms, fname_i_worms)

# output
# -----
dummy_cell_arg_o = "dummy output"

dir_wrims = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_wrims):
    os.makedirs(dir_wrims)

fname_wrims_o = 'classification.csv'
file_wrims_o  = os.path.join(dir_wrims, fname_wrims_o)

# func
# -----

# start
# -----
# Get PEMA parameters file from github
# .....
url_github_csv = "https://raw.githubusercontent.com/arms-mbon/data_workspace/main/reformatted_data/lifewatch/ARMS4Tesseract_PEMA_data.csv"

fname_github_tsv = "ARMS4Tesseract_PEMA_data.tsv"
file_github_tsv  = os.path.join(dir_wrims, fname_github_tsv)

obj_response = requests.get(url_github_csv)
if obj_response.status_code != 200:
    raise RuntimeError('CSV file not found in ' + url_github_csv)

fobj_url_github_csv = csv.reader(StringIO(obj_response.text), delimiter=conf_delimiter_csv)

with open(file_github_tsv, 'w') as fp_tsv:
    writer = csv.writer(fp_tsv, delimiter=conf_delimiter_tsv)
    
    for row in fobj_url_github_csv:
        writer.writerow(row)

# Run WRiMS Invasive Checker
# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_exec = f'python "{file_lib_InvasiveChecker}" -i "{file_i_worms}" -m "{file_github_tsv}" -o "{dir_wrims}"'
    print(str_exec)
    fp_log.write(f"\n```\n{str_exec}\n```\n")

    os.system(str_exec)

# Copy files
# shutil.copyfile(f"{dir_wrims}/final_table_WoRMSandWRIMS.csv", "/mnt/outputs/final_table_WoRMSandWRiMS.csv")
# shutil.copyfile(f"{dir_wrims}/ARMS4Tesseract_PEMA_data.csv",  "/mnt/outputs/ARMS4Tesseract_PEMA_data.csv")
# shutil.copyfile(f"{dir_wrims}/OTU_withAphia.csv",             "/mnt/outputs/OTU_withAphia.csv")

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    df_wrims = pd.read_csv(file_wrims_o, sep=conf_delimiter_csv)
    
    str_print = f"{df_wrims}"
    print(str_print)
    fp_log.write(f"\n```\n{str_print}\n```\n\n")

    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {dir_wrims}\n")

print(f"Finish: {workflow_step}")


test
python "/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/invasive_checker/main.py" -i "/home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/WoRMS/worms.tsv" -m "/home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/WRiMS/ARMS4Tesseract_PEMA_data.tsv" -o "/home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/WRiMS"


2026-04-09 08:47:16,898 - INFO - main - Input file: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/WoRMS/worms.tsv
2026-04-09 08:47:16,899 - INFO - main - Output folder: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/WRiMS
2026-04-09 08:47:16,899 - INFO - main - Input metadata File: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-test/WRiMS/ARMS4Tesseract_PEMA_data.tsv
2026-04-09 08:47:16,899 - INFO - main - Extra config: {
  "LLEVEL": "INFO",
  "ID_SOURCE": "sciname",
  "SEP": "\t",
  "OTU_COL_NAME": "OTU",
  "CLASS_COL_NAME": "classification",
  "CLASS_OUTPUT_FILE": "classification.csv",
  "WORMS_OUTPUT_FILE": "worms.csv"
}
2026-04-09 08:47:16,899 - INFO - main -   -Preparing data...
2026-04-09 08:47:16,913 - INFO - main - Index(['OTU', 'SRR3231901', 'classification', 'aphia_id', 'location_id'], dtype='str')
2026-04-09 08:47:16,915 - INFO - main - Index(['OTU', 'SRR3231901', 'classification'], dtype='str')
2026-04-09 08:47:16,916 - INFO - main - Done pre

  classification\tOTU\tAccessionID\tCount\tisNegativeControlGene\tGeneType\tlatitude\tlongitude\tSample_ID\tAphia_ID\tWorms SciName\tWorms SciName Rank\tWRIMS Status at Sample Location\tMarineRegions with known occurrence at Sample Location
0  Main genome;Eukaryota;SAR;Stramenopiles;Ochrop...                                                                                                                                                                                            
1  Main genome;Eukaryota;Cryptophyceae;Cryptomona...                                                                                                                                                                                            
2  Chloroplast;Eukaryota (Chloroplast);Archaeplas...                                                                                                                                                                                            
3  Chloroplast;Eukaryota (Chloroplas

In [9]:
# DNA, PEMA-Converter
# =====
# use R scripot from R kernal ipynb

# workflow: 01, 03
# -----
# print("Finish: PEMA-Converter")


In [10]:
# DNA, metamds-observations
# =====
# use R scripot from R kernal ipynb

# workflow: 01
# -----
# print("Finish: metamds-observations")


In [11]:
# DNA, Trend Analysis
# =====

# workflow: 02
# -----


In [12]:
# DNA, Taxa2Dis
# =====

# workflow: 03
# -----


In [13]:
# DNA, Taxondive
# =====

# workflow: 03
# -----


In [14]:
# DNA, workflow finish
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
# ...

import os
import sys
from datetime import datetime

# sys.path.append(conf_minio_public_local_code)
# sys.path.append(conf_minio_user_local_code)

print(param_workflow_name)
print(param_gene_sequences)
print(param_fname_par_tsv)
workflow_step = f"{conf_vlab_name}-Finish"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")


test
SRR3231901
Template-parameters.tsv
Finish: DNA-Finish
